# Synthetic Document Generation

Generate synthetic receipts and invoices as PDFs for multimodal AI function examples. These documents enable:
- **Document classification** — classify images as receipt vs invoice using `AI.GENERATE` or `AI.CLASSIFY`
- **Information extraction** — use `AI.GENERATE` with `output_schema` to pull structured data from receipts
- **Document processing** — use `ML.PROCESS_DOCUMENT` for invoices

**What this notebook does:**
1. Generates realistic receipt and invoice data using `AI.GENERATE` with `output_schema`
2. Renders HTML templates to PDF using weasyprint
3. Builds a ground truth manifest with generic file naming (`doc_001.pdf` through `doc_100.pdf`)

GCS upload and object table creation happen in the downstream workflow notebooks that use these documents.

**Prerequisites:** A GCP project with BigQuery enabled. See the [Setup Reference](../../setup/) for details.

---
## Configuration

Set your project and generation parameters.

In [1]:
PROJECT_ID = 'statmike-mlops-349915'  # <-- Replace with your project ID
LOCATION = 'US'  # BigQuery location (for AI.GENERATE queries)

In [2]:
# Generation parameters
N_RECEIPTS = 50
N_INVOICES = 50
SEED = 42  # Used for manifest shuffle (random doc_NNN.pdf assignment)

### Environment

> **Already set up the project environment?** The cell below is a no-op — packages are already in your kernel. See the [Setup Reference](../../setup/) for details.
>
> **Running standalone** (Colab, Colab Enterprise, Vertex AI Workbench)? The cell below installs required packages into your current kernel.

In [3]:
import subprocess, sys, shutil

def install(*packages):
    """Install packages using uv (fast) with pip fallback."""
    uv = shutil.which('uv')
    if uv:
        subprocess.check_call([uv, 'pip', 'install', '-q', '--python', sys.executable, *packages])
    else:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', *packages])

install(
    'google-cloud-bigquery', 'db-dtypes',
    'pydantic', 'Jinja2', 'weasyprint',
)

In [4]:
# Authenticate (Colab only — other environments use Application Default Credentials)
try:
    from google.colab import auth
    auth.authenticate_user()
except ImportError:
    pass  # Not in Colab — ADC is used automatically

In [ ]:
from pathlib import Path
from google.cloud import bigquery

client = bigquery.Client(project=PROJECT_ID)
print(f'BigQuery client ready (project: {PROJECT_ID})')

# Resolve paths — works both from repo checkout and Colab
try:
    NOTEBOOK_DIR = Path(__file__).parent
except NameError:
    NOTEBOOK_DIR = Path.cwd()
    if not (NOTEBOOK_DIR / 'modules').exists():
        for candidate in [Path('data/documents'), Path('.')]:
            if (candidate / 'modules').exists():
                NOTEBOOK_DIR = candidate
                break
        else:
            raise FileNotFoundError(
                'Cannot find modules/ directory. '
                'Run this notebook from the data/documents/ directory, '
                'or clone the repo first: '
                'git clone https://github.com/statmike/vertex-ai-mlops.git'
            )

TEMPLATE_DIR = NOTEBOOK_DIR / 'templates'
RECEIPTS_DIR = NOTEBOOK_DIR / 'receipts'
INVOICES_DIR = NOTEBOOK_DIR / 'invoices'

RECEIPTS_DIR.mkdir(exist_ok=True)
INVOICES_DIR.mkdir(exist_ok=True)

sys.path.insert(0, str(NOTEBOOK_DIR))

print(f'Template dir: {TEMPLATE_DIR}')
print(f'Output dirs: {RECEIPTS_DIR}, {INVOICES_DIR}')

---
## Step 1 — Generate Data

Use `AI.GENERATE` to create realistic receipt and invoice data. The data structures are defined as Pydantic models in [`modules/schemas.py`](modules/schemas.py) — reusable by any notebook that needs to generate or validate documents.

**Approach:** The `output_schema` is a single `json STRING` field. The prompt describes the expected JSON structure (generated from the Pydantic model field descriptions), and Gemini returns a JSON string that we parse in Python.

In [6]:
import json
from modules.schemas import Receipt, Invoice, json_schema_prompt

# Build prompt instructions from the Pydantic models
receipt_prompt = json_schema_prompt(Receipt)
invoice_prompt = json_schema_prompt(Invoice)

print('Receipt JSON schema (from Pydantic):')
print(receipt_prompt)

# Escape for SQL string literals (BigQuery single-quoted strings can't contain raw newlines)
receipt_prompt_sql = receipt_prompt.replace('\n', '\\n')
invoice_prompt_sql = invoice_prompt.replace('\n', '\\n')

def parse_json_response(raw):
    """Parse JSON from Gemini, unwrapping array-wrapped responses."""
    parsed = json.loads(raw)
    if isinstance(parsed, list):
        parsed = parsed[0]
    return parsed

# --- Generate receipts ---
print(f'\nGenerating {N_RECEIPTS} receipts via AI.GENERATE...')
receipt_query = f'''
SELECT n, (AI.GENERATE(
  CONCAT(
    'Generate realistic US store receipt data as a JSON object. Make receipt #', CAST(n AS STRING),
    ' unique — vary the store name, store type (grocery, pharmacy, hardware, convenience, ',
    'deli, bakery, pet supply, electronics), location, number of items, and payment method. ',
    'Return a JSON object with these keys:\\n{receipt_prompt_sql}'
  ),
  endpoint => 'gemini-2.5-flash',
  output_schema => 'json STRING'
)).json AS receipt_json
FROM UNNEST(GENERATE_ARRAY(1, {N_RECEIPTS})) AS n
'''
df_receipts = client.query(receipt_query).to_dataframe()
receipts = []
for _, row in df_receipts.iterrows():
    r = parse_json_response(row['receipt_json'])
    for key in ['subtotal', 'tax_rate', 'tax', 'total', 'cash_tendered', 'change']:
        if key in r and r[key] is not None:
            r[key] = float(r[key])
    for item in r.get('items', []):
        for k in ['qty', 'price', 'total']:
            if k in item:
                item[k] = float(item[k]) if k != 'qty' else int(item[k])
    receipts.append(r)
print(f'  {len(receipts)} receipts generated')

# --- Generate invoices ---
print(f'\nGenerating {N_INVOICES} invoices via AI.GENERATE...')
invoice_query = f'''
SELECT n, (AI.GENERATE(
  CONCAT(
    'Generate realistic US business invoice data as a JSON object. Make invoice #', CAST(n AS STRING),
    ' unique — vary the company name and type, client company, services offered, ',
    'dates, and payment terms. ',
    'Return a JSON object with these keys:\\n{invoice_prompt_sql}'
  ),
  endpoint => 'gemini-2.5-flash',
  output_schema => 'json STRING'
)).json AS invoice_json
FROM UNNEST(GENERATE_ARRAY(1, {N_INVOICES})) AS n
'''
df_invoices = client.query(invoice_query).to_dataframe()
invoices = []
for _, row in df_invoices.iterrows():
    inv = parse_json_response(row['invoice_json'])
    for key in ['subtotal', 'tax_rate', 'tax', 'total_due']:
        if key in inv and inv[key] is not None:
            inv[key] = float(inv[key])
    for item in inv.get('line_items', []):
        for k in ['qty', 'unit_price', 'amount']:
            if k in item:
                item[k] = float(item[k]) if k != 'qty' else int(item[k])
    invoices.append(inv)
print(f'  {len(invoices)} invoices generated')

# Show one example of each
print(f'\n--- Example Receipt ---')
print(f'Store: {receipts[0]["store_name"]}')
print(f'Items: {len(receipts[0]["items"])}')
print(f'Total: ${receipts[0]["total"]:.2f}')
print(f'Payment: {receipts[0]["payment_method"]}')

print(f'\n--- Example Invoice ---')
print(f'Company: {invoices[0]["company_name"]}')
print(f'Client: {invoices[0]["client_name"]}')
print(f'Invoice #: {invoices[0]["invoice_number"]}')
print(f'Line items: {len(invoices[0]["line_items"])}')
print(f'Total due: ${invoices[0]["total_due"]:.2f}')

Receipt JSON schema (from Pydantic):
- "store_name": (string) — Unique store name
- "address": (string) — US street address
- "city_state_zip": (string) — City, state abbreviation, and ZIP
- "phone": (string) — Phone number with area code
- "date": (string) — Date in MM/DD/YYYY format, within the past year
- "time": (string) — Time in HH:MM AM/PM format
- "items": array of objects — 3-15 purchased items, each with:
  - "name": (string) — Short product name like a thermal receipt printout
  - "qty": (integer) — Quantity purchased, 1-4
  - "price": (number) — Unit price, $0.99-$14.99
  - "total": (number) — Line total: qty * price
- "subtotal": (number) — Sum of all item totals
- "tax_rate": (number) — Tax rate as decimal between 0.04 and 0.10
- "tax": (number) — Tax amount: round(subtotal * tax_rate, 2)
- "total": (number) — Grand total: subtotal + tax
- "payment_method": (string) — CASH or card type with last 4 digits like VISA ****1234
- "cash_tendered": (number or null) — Cash tender

---
## Step 2 — Render PDFs

Convert each data dictionary to a PDF using Jinja2 HTML templates and weasyprint. Receipts use a thermal printer style (narrow, monospace), invoices use a professional business layout.

In [7]:
from modules.renderers import render_receipt, render_invoice
from modules.styles import receipt_style, invoice_style

# Render receipts — each gets a unique visual style seeded by index
for i, data in enumerate(receipts):
    style = receipt_style(seed=i)
    pdf_bytes = render_receipt(data, TEMPLATE_DIR, style=style)
    path = RECEIPTS_DIR / f'receipt_{i+1:03d}.pdf'
    path.write_bytes(pdf_bytes)
print(f'Saved {N_RECEIPTS} receipts to {RECEIPTS_DIR}/')

# Render invoices — each gets a unique visual style seeded by index
for i, data in enumerate(invoices):
    style = invoice_style(seed=i)
    pdf_bytes = render_invoice(data, TEMPLATE_DIR, style=style)
    path = INVOICES_DIR / f'invoice_{i+1:03d}.pdf'
    path.write_bytes(pdf_bytes)
print(f'Saved {N_INVOICES} invoices to {INVOICES_DIR}/')

# Show file sizes for a quick sanity check
sample_receipt = RECEIPTS_DIR / 'receipt_001.pdf'
sample_invoice = INVOICES_DIR / 'invoice_001.pdf'
print(f'\nSample receipt size: {sample_receipt.stat().st_size / 1024:.1f} KB')
print(f'Sample invoice size: {sample_invoice.stat().st_size / 1024:.1f} KB')

Saved 50 receipts to /usr/local/google/home/statmike/Git/google_projects/projects/bq-ai-functions/data/documents/receipts/
Saved 50 invoices to /usr/local/google/home/statmike/Git/google_projects/projects/bq-ai-functions/data/documents/invoices/

Sample receipt size: 12.7 KB
Sample invoice size: 16.8 KB


In [8]:
from IPython.display import display, HTML
from jinja2 import Environment, FileSystemLoader
import html as html_lib

env = Environment(loader=FileSystemLoader(str(TEMPLATE_DIR)))

# Preview 6 receipts — showing layout and style variety
display(HTML('<h3>Sample Receipts (varied layouts &amp; styles)</h3>'))
frames = ''
for i in range(6):
    style = receipt_style(seed=i)
    rendered = env.get_template(style['template']).render(**receipts[i], style=style)
    escaped = html_lib.escape(rendered)
    frames += f'<iframe srcdoc="{escaped}" width="300" height="550" style="border:1px solid #ccc; margin:5px;"></iframe>'
display(HTML(f'<div style="display:flex; gap:10px; flex-wrap:wrap;">{frames}</div>'))

# Preview 6 invoices — showing layout and style variety
display(HTML('<h3>Sample Invoices (varied layouts &amp; styles)</h3>'))
frames = ''
for i in range(6):
    style = invoice_style(seed=i)
    rendered = env.get_template(style['template']).render(**invoices[i], style=style)
    escaped = html_lib.escape(rendered)
    frames += f'<iframe srcdoc="{escaped}" width="520" height="650" style="border:1px solid #ccc; margin:5px;"></iframe>'
display(HTML(f'<div style="display:flex; gap:10px; flex-wrap:wrap;">{frames}</div>'))

---
## Step 3 — Build Manifest

Create a ground truth manifest that maps generic file names (`doc_001.pdf` through `doc_100.pdf`) to source files and key fields. The generic naming ensures that file names don't reveal document type — classification becomes the real first step in any downstream pipeline.

The shuffle uses the same `SEED` for reproducibility.

In [9]:
import random

# Build list of all documents with their source info
all_docs = []

for i, data in enumerate(receipts):
    all_docs.append({
        'type': 'receipt',
        'source': f'receipt_{i+1:03d}.pdf',
        'key_fields': {
            'store_name': data['store_name'],
            'total': data['total'],
            'date': data['date'],
        },
    })

for i, data in enumerate(invoices):
    all_docs.append({
        'type': 'invoice',
        'source': f'invoice_{i+1:03d}.pdf',
        'key_fields': {
            'company': data['company_name'],
            'invoice_number': data['invoice_number'],
            'total_due': data['total_due'],
            'due_date': data['due_date'],
        },
    })

# Shuffle with fixed seed so generic names are randomly assigned
random.seed(SEED)
random.shuffle(all_docs)

# Assign generic names and build manifest
manifest = {}
for idx, doc in enumerate(all_docs):
    generic_name = f'doc_{idx+1:03d}.pdf'
    manifest[generic_name] = {
        'type': doc['type'],
        'source': doc['source'],
        'key_fields': doc['key_fields'],
    }

# Save manifest locally
manifest_path = NOTEBOOK_DIR / 'manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2))

# Summary
n_receipts_mapped = sum(1 for v in manifest.values() if v['type'] == 'receipt')
n_invoices_mapped = sum(1 for v in manifest.values() if v['type'] == 'invoice')
print(f'Manifest: {len(manifest)} documents ({n_receipts_mapped} receipts, {n_invoices_mapped} invoices)')
print(f'Saved to {manifest_path}')

# Show first few mappings
print('\nSample mappings:')
for name, info in list(manifest.items())[:5]:
    print(f'  {name} -> {info["source"]} ({info["type"]})')

Manifest: 100 documents (50 receipts, 50 invoices)
Saved to /usr/local/google/home/statmike/Git/google_projects/projects/bq-ai-functions/data/documents/manifest.json

Sample mappings:
  doc_001.pdf -> receipt_043.pdf (receipt)
  doc_002.pdf -> receipt_042.pdf (receipt)
  doc_003.pdf -> invoice_042.pdf (invoice)
  doc_004.pdf -> receipt_010.pdf (receipt)
  doc_005.pdf -> invoice_016.pdf (invoice)
